# 02 — Attribution

Loads pre-extracted data, runs all registered attribution methods, and saves
results as a timestamped CSV in `results/`.

## How to add your own method
1. Implement it in the **Custom methods** cell below (a template is provided).
2. Register it in `ATTRIBUTION_METHODS` with a unique key.
3. Re-run the pipeline cells — your method will appear automatically in the output CSV.

The `ctx` dict available inside each lambda contains:
```
ctx['tas_f']    (T,)             factual temperature time series
ctx['gmt_f']    (T,)             smoothed factual GMT anomaly
ctx['gmt_c']    (T,)             smoothed counterfactual GMT
ctx['slp_f']    (T, n_lat, n_lon) global SLP field — slice as needed
ctx['slp_lat']  (n_lat,)         latitude axis
ctx['slp_lon']  (n_lon,)         longitude axis
ctx['ev_lat']   float            event barycentre latitude
ctx['ev_lon']   float            event barycentre longitude
ctx['val']      float            event threshold
ctx['range']    (t0, t1)         PN computation window
ctx['t_obs']    int              event time index
ctx['mth']      str              PN estimator set at runtime
```

In [1]:
import sys, os, pickle
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed

SRC_PATH = '/content/drive/MyDrive/ellis-attribution/src'
sys.path.insert(0, SRC_PATH)

from src.attribution import (
    compute_pn,
    run_thermo_ml,
    run_dyn_adj_local,
    run_dml_dyn_adj_local
)
from src.analogues import (
    run_analogues,
    run_analogues_local,
    run_analogues_local_lasso,
    run_analogues_causal_knn,
    run_knn_attributor
)
from src.data_utils import extract_local_slp

/home/homer/anaconda3/envs/attribution_chanllenge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-24 21:19:33,285	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# ================================================================
#  CONFIG  —  edit here
# ================================================================
path_extracted = '/home/homer/Documents/Research/Data/attribution_evaluation_data/extracted/'
DATA_PATH    = path_extracted + 'extracted_tasmax_nmem10_start2004_p99.9.pkl'

RESULTS_DIR = 'results'

# PN estimators to run for every method
# Options: 'empirical', 'gaussian', 'gev'  (can select multiple)
STAT_METHODS = ['gaussian']

# Time window around each event
WINDOW_BEFORE = 72   # months
WINDOW_AFTER  = 12   # months

N_JOBS = 2#os.cpu_count()
# ================================================================

## Built-in methods
These call functions from `src/attribution.py` and `src/analogues.py`.
Do not modify this cell — add your own methods in the **Custom methods** cell below.

In [3]:
ATTRIBUTION_METHODS = {

    # ── Thermodynamic ───────────────────────────────────────────────────────
    'pn_thermo_ML': lambda ctx:
        run_thermo_ml(
            ctx['tas'], ctx['gmt'],
            ctx['val'], ctx['range'], ctx['mth']),

    # ── Analogues —  local SLP box ───────────────────────────
    'pn_analogues_local': lambda ctx:
        run_analogues_local(
            ctx['tas'], ctx['slp'],
            ctx['slp_lat'], ctx['slp_lon'],
            ctx['ev_lat'], ctx['ev_lon'],
            ctx['val'], ctx['t_obs'], ctx['range'], ctx['mth'],
            half_width_deg=50.0, n_years=50, n_analogues=50),
        
    # ── Dynamical Adjustment - local SLP box ─────────────────────
    'pn_dynamical_adjustment_local': lambda ctx:
        run_dyn_adj_local(ctx['tas'], ctx['slp'],
            ctx['slp_lat'], ctx['slp_lon'],
            ctx['ev_lat'], ctx['ev_lon'],
            ctx['val'], ctx['range'], ctx['mth'],
            half_width_deg=25.0, n_years=50),

        
    # ── KNN Attributor (dual-world, Lasso PCA features) ─────────────────────
    'pn_analogue_sparse': lambda ctx:
        run_knn_attributor(
            ctx['tas'],
            ctx['slp'],
            ctx['slp_lat'], ctx['slp_lon'],
            ctx['ev_lat'], ctx['ev_lon'],
            ctx['val'], ctx['t_obs'], ctx['range'], ctx['mth'],
            half_width_deg=25.0,
            lasso_method='sir', n_analogues=25),

}

In [4]:
def build_ctx(res, e_idx, is_factual):
    """Assemble the context dict passed to every attribution method."""
    tas_run = res['f_tas'] if is_factual else res['c_tas']
    slp_run = res['f_slp'] if is_factual else res['c_slp']
    idx_run = res['idx_f'] if is_factual else res['idx_c']
    t_obs   = idx_run[e_idx]
    t0      = max(0, t_obs - WINDOW_BEFORE)
    t1      = min(tas_run.shape[0], t_obs + WINDOW_AFTER)
    return {
        'tas':   tas_run[:, e_idx],
        'gmt':   res['gmt4_f'] if is_factual else res['gmt4_c'],
        'slp':   slp_run,
        'slp_lat': res['slp_lat'],
        'slp_lon': res['slp_lon'],
        'ev_lat':  res['location'][e_idx, 0],
        'ev_lon':  res['location'][e_idx, 1],
        'val':     res['f_tas_vals'][e_idx] if is_factual else res['c_tas_vals'][e_idx],
        'range':   (t0, t1),
        't_obs':   t_obs,
    }


def process_event(m_idx, is_factual, data_list):
    """Run all methods on every event of one member-scenario."""
    res      = data_list[m_idx]
    scenario = 'factual' if is_factual else 'counterfactual'
    n_events = res['f_tas'].shape[1]
    rows     = []

    for e_idx in range(n_events):
        ctx = build_ctx(res, e_idx, is_factual)

        # Ensemble ground truth
        pool_f = np.concatenate([
            data_list[mi]['f_tas'][ctx['range'][0]:ctx['range'][1], e_idx]
            for mi in range(len(data_list))
            if e_idx < data_list[mi]['f_tas'].shape[1]
        ])
        pool_c = np.concatenate([
            data_list[mi]['c_tas'][ctx['range'][0]:ctx['range'][1], e_idx]
            for mi in range(len(data_list))
            if e_idx < data_list[mi]['c_tas'].shape[1]
        ])

        row = {
            'member':   res['member'],
            'event_id': e_idx,
            'scenario': scenario,
            'time':     res['times'][ctx['t_obs']],
            'lat':      ctx['ev_lat'],
            'lon':      ctx['ev_lon'],
        }

        for mth in STAT_METHODS:
            row[f'pn_ensemble_{mth}'] = compute_pn(pool_f, pool_c, ctx['val'], method=mth)
            for col_name, func in ATTRIBUTION_METHODS.items():
                ctx['mth'] = mth
                try:
                    row[f'{col_name}_{mth}'] = func(ctx)
                except Exception as exc:
                    print(f'[{res["member"]} e{e_idx} {mth} {col_name}] {exc}')
                    row[f'{col_name}_{mth}'] = np.nan

        rows.append(row)
    return rows

In [5]:
with open(DATA_PATH, 'rb') as fh:
    data = pickle.load(fh)
print(f'Loaded {len(data)} members, {data[0]["f_tas"].shape[1]} events each.')

tasks   = [(i, fact) for i in range(len(data)) for fact in (True, False)]
results = Parallel(n_jobs=N_JOBS)(
    delayed(process_event)(m, f, data) for m, f in tqdm(tasks)
)

df = pd.DataFrame([row for sublist in results for row in sublist])
df['time'] = pd.to_datetime(df['time'])
print(f'Results shape: {df.shape}')
df.head()

Loaded 10 members, 29 events each.


  0%|          | 0/20 [00:00<?, ?it/s]

[r1i1p1f1 e0 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r1i1p1f1 e0 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r1i1p1f1 e1 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r1i1p1f1 e1 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r1i1p1f1 e2 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r1i1p1f1 e2 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r1i1p1f1 e3 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r1i1p1f1 e3 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]


 20%|██        | 4/20 [00:25<01:41,  6.37s/it]

[r1i1p1f1 e28 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r2i1p1f1 e0 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r2i1p1f1 e1 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r2i1p1f1 e0 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r2i1p1f1 e2 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r2i1p1f1 e1 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r2i1p1f1 e3 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r2i1p1f1 e2 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]

...

 90%|█████████ | 18/20 [01:20<00:08,  4.18s/it]

[r8i1p1f1 e17 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r9i1p1f1 e0 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r8i1p1f1 e18 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r9i1p1f1 e1 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r8i1p1f1 e19 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r9i1p1f1 e2 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r9i1p1f1 e0 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r9i1p1f1 e3 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 198

100%|██████████| 20/20 [01:26<00:00,  4.31s/it]


[r9i1p1f1 e13 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r10i1p1f1 e0 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r9i1p1f1 e14 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r10i1p1f1 e1 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r10i1p1f1 e0 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r10i1p1f1 e2 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r10i1p1f1 e1 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600, 1980]
[r10i1p1f1 e3 gaussian pn_dynamical_adjustment_local] Found input variables with inconsistent numbers of samples: [600

,member,event_id,scenario,time,lat,lon,pn_ensemble_gaussian,pn_thermo_ML_gaussian,pn_analogues_local_gaussian,pn_dynamical_adjustment_local_gaussian,pn_analogue_sparse_gaussian
0,r1i1p1f1,0,factual,2004-02-15 12:00:00,43.774647,50.750000,0.868749,0.928612,0.299978,NaN,0.799552
1,r1i1p1f1,1,factual,2004-06-16 00:00:00,-72.351028,43.230770,0.720876,0.761629,0.511571,NaN,0.999936
2,r1i1p1f1,2,factual,2004-12-16 12:00:00,63.058010,151.652542,0.888448,0.995977,0.999999,NaN,0.999998
3,r1i1p1f1,3,factual,2005-05-16 12:00:00,-19.544710,21.744186,0.642460,0.995962,1.000000,NaN,1.000000
4,r1i1p1f1,4,factual,2006-04-16 00:00:00,51.557922,75.000000,0.977754,0.872158,0.000000,NaN,0.996334


In [6]:
os.makedirs(RESULTS_DIR, exist_ok=True)
timestamp  = datetime.now().strftime('%Y%m%d_%H%M')
save_path  = os.path.join(RESULTS_DIR, f'attribution_{timestamp}.csv')
df.to_csv(save_path, index=False)
print(f'Saved -> {save_path}')

Saved -> results/attribution_20260324_2121.csv
